### 데이터 다운로드

In [1]:
from sklearn.datasets import fetch_20newsgroups
import pandas as pd

ng = fetch_20newsgroups(subset="all", remove=())  # remove=() → 아무것도 제거하지 않음 (헤더/풋터/인용 유지)
df_20ng = pd.DataFrame({
    "text": ng.data,
    "label_id": ng.target,
    "label_name": [ng.target_names[i] for i in ng.target]
})
print(df_20ng.shape, df_20ng.head(2))


(18846, 3)                                                 text  label_id  \
0  From: Mamatha Devineni Ratnam <mr47+@andrew.cm...        10   
1  From: mblawson@midway.ecn.uoknor.edu (Matthew ...         3   

                 label_name  
0          rec.sport.hockey  
1  comp.sys.ibm.pc.hardware  


In [3]:
df_20ng['text'][0]

"From: Mamatha Devineni Ratnam <mr47+@andrew.cmu.edu>\nSubject: Pens fans reactions\nOrganization: Post Office, Carnegie Mellon, Pittsburgh, PA\nLines: 12\nNNTP-Posting-Host: po4.andrew.cmu.edu\n\n\n\nI am sure some bashers of Pens fans are pretty confused about the lack\nof any kind of posts about the recent Pens massacre of the Devils. Actually,\nI am  bit puzzled too and a bit relieved. However, I am going to put an end\nto non-PIttsburghers' relief with a bit of praise for the Pens. Man, they\nare killing those Devils worse than I thought. Jagr just showed you why\nhe is much better than his regular season stats. He is also a lot\nfo fun to watch in the playoffs. Bowman should let JAgr have a lot of\nfun in the next couple of games since the Pens are going to beat the pulp out of Jersey anyway. I was very disappointed not to see the Islanders lose the final\nregular season game.          PENS RULE!!!\n\n"

### 기본 전처리

In [4]:
# 기본 전처리 -> 필요한 내용만 남기고 없애기
import re

HEADER_RE = re.compile(r"^(?:[A-Za-z\-]+:.*\n)+\n", re.MULTILINE)  # 헤더블록: 첫 빈 줄까지
SIG_RE    = re.compile(r"\n-- ?\n.*", re.DOTALL)                    # 서명 블록(옵션)
RE_EMAIL = re.compile(r"(?i)\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b")

def preprocess_20ng_raw(text: str,
                        keep_quotes=False) -> str:
    # 1) 헤더 제거
    text = re.sub(HEADER_RE, "", text)

    # 2) 인용 줄 제거(옵션)
    if not keep_quotes:
        text = "\n".join(line for line in text.splitlines() if not line.strip().startswith(">"))

    text = re.sub(SIG_RE, "", text)
    text = re.sub(RE_EMAIL, "", text)

    # 4) 소문자화
    text = text.lower()

    # 5) 구두점/특수문자 제거(아포스트로피 포함 → 'party s' 같은 형태가 됨)
    text = re.sub(r"[^a-z0-9]+", " ", text)

    # 6) 공백 정규화
    text = re.sub(r"\s+", " ", text).strip()

    return text


In [5]:
df_20ng['text'] = df_20ng['text'].apply(preprocess_20ng_raw)

In [6]:
df_20ng.head()

,text,label_id,label_name
0,i am sure some bashers of pens fans are pretty...,10,rec.sport.hockey
1,my brother is in the market for a high perform...,3,comp.sys.ibm.pc.hardware
2,the student of regional killings alias davidia...,17,talk.politics.mideast
3,in article wayne smith writes think it s the s...,3,comp.sys.ibm.pc.hardware
4,1 i have an old jasmine drive which i cannot u...,4,comp.sys.mac.hardware


In [7]:
df_20ng['text'][0]

'i am sure some bashers of pens fans are pretty confused about the lack of any kind of posts about the recent pens massacre of the devils actually i am bit puzzled too and a bit relieved however i am going to put an end to non pittsburghers relief with a bit of praise for the pens man they are killing those devils worse than i thought jagr just showed you why he is much better than his regular season stats he is also a lot fo fun to watch in the playoffs bowman should let jagr have a lot of fun in the next couple of games since the pens are going to beat the pulp out of jersey anyway i was very disappointed not to see the islanders lose the final regular season game pens rule'

### 이상한 데이터 drop

In [8]:
suspects = ["nrhj", "wwiz", "gizw", "bhjn", "pmfq", "ax", "subscrive", "subscribe"]

In [9]:
def drop_suspect_docs(df: pd.DataFrame, text_col: str = "text", suspects=None):
    if suspects is None:
        return df, pd.DataFrame()

    # 소문자 기준 검사
    suspects_lower = [w.lower() for w in suspects]

    mask_suspect = df[text_col].str.split().apply(
        lambda toks: any(w in suspects_lower for w in toks)
        if isinstance(toks, list) else False
    )

    dropped_df = df[mask_suspect].copy()       # drop된 행
    kept_df    = df[~mask_suspect].copy()      # 남은 행

    print(f"[drop_suspect_docs] dropped {dropped_df.shape[0]} / {df.shape[0]} rows")
    return kept_df, dropped_df

# 사용 예시
df_20ng, dropped_df = drop_suspect_docs(df_20ng, text_col="text", suspects=suspects)

# drop된 행 보기
print(dropped_df.head())

[drop_suspect_docs] dropped 104 / 18846 rows
                                                   text  label_id  \
174                                 please subscrive me         5   
335                                           subscribe         5   
546   lightwave3d mail list what is lightwave lightw...         1   
717   in article neil b gandler writes that depends ...        12   
1121  in article tom van flandern writes i hold that...        14   

           label_name  
174    comp.windows.x  
335    comp.windows.x  
546     comp.graphics  
717   sci.electronics  
1121        sci.space  


### 강한 전처리

In [10]:
# pip install regex scikit-learn
# (선택) pip install spacy && python -m spacy download en_core_web_sm

import regex as re
import unicodedata
from collections import Counter
from typing import Iterable, List, Set, Optional, Dict, Any
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# ======================
# Regex patterns
# ======================
RE_URL    = re.compile(r"(?i)\b(?:https?://|www\.)\S+\b")
RE_EMAIL  = re.compile(r"(?i)\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b")
RE_PUNCT  = re.compile(r"[^\p{L}\s]")          # 문자/공백 외 제거
RE_TOKEN  = re.compile(r"\b\p{L}+\b")          # 문자 토큰
RE_PATH   = re.compile(r"(?m)(?:^|[\s])(?:/[\w\-.]+)+")  # 유닉스 경로 대충
RE_CODE   = re.compile(r"[`]{1,3}.*?[`]{1,3}", re.S)     # 코드 블록/인라인
RE_SIG    = re.compile(r"(?ms)^\s*--\s*$.*")             # '--'부터 끝까지 서명
RE_PGP    = re.compile(r"(?ms)^-+BEGIN PGP.*?END PGP-+$")
RE_QUOTES = re.compile(
    r"(?mi)^(?:\s*>.*|in article .* writes:.*|on .* wrote:.*|^\|.*)$"
)  # 인용/답장 패턴 줄 단위 제거

HEADER_PREFIXES = ("from:", "organization:", "nntp-posting-host:", "path:", "reply-to:", "subject:", "lines:")

# ======================
# 헤더 전용 stopword 채집
# ======================
def mine_header_stopwords(texts: Iterable[str], top_k: int = 150) -> Set[str]:
    """본문은 건드리지 않고, '헤더 줄'에서만 자주 보이는 토큰을 stopwords로 수집."""
    c = Counter()
    for t in texts:
        if not isinstance(t, str):
            continue
        for ln in t.splitlines():
            ls = ln.strip().lower()
            if not ls.startswith(HEADER_PREFIXES):
                continue
            ls = RE_URL.sub(" ", ls)
            ls = RE_EMAIL.sub(" ", ls)
            ls = RE_PUNCT.sub(" ", ls)
            for tok in RE_TOKEN.findall(ls):
                if len(tok) < 2:
                    continue
                c[tok] += 1
    base_sw = set(ENGLISH_STOP_WORDS)
    header_sw = [w for w, _ in c.most_common(top_k) if w not in base_sw]
    return set(header_sw)

# ======================
# 불용어 빌더 (기본 + 헤더 + 메타)
# ======================
REPORTING_STOP = {
    # 보고/메타 단어 (lemmatize 전/후를 모두 포괄)
    "write","writes","wrote","say","says","said","think","thinks","believe","believes","know","knows",
    "like","likes","just","good","right","people","anyone","someone","thanks","thank",
    "mail","list","information","address","university","internet","send","using","article","subject"
}

META_STOP = {
    # 헤더 날짜/도메인 꼬리/국가코드 등
    "jan","feb","mar","apr","may","jun","jul","aug","sep","oct","nov","dec",
    "com","org","net","gov","edu","mil","rri",
    "us","usa","uk","ca","au","de","fr","jp","kr","cn","it","es","se",
    "tel","fax","phone","mobile","contact",
}

def build_stopwords(header_texts: Optional[Iterable[str]] = None,
                    extra_stopwords: Optional[Iterable[str]] = None) -> Set[str]:
    sw = set(ENGLISH_STOP_WORDS)
    if header_texts is not None:
        sw |= mine_header_stopwords(header_texts, top_k=150)
    sw |= REPORTING_STOP
    sw |= META_STOP
    if extra_stopwords:
        sw |= set(map(str.lower, extra_stopwords))
    return sw

# ======================
# 20NG 전용 노이즈 스트리핑
# ======================
def strip_20ng_noise(text: str) -> str:
    """헤더/인용/서명/PGP/코드/경로/URL/이메일 등을 정리하고 공백 정규화."""
    if not isinstance(text, str):
        return ""
    t = unicodedata.normalize("NFKC", text)
    # 코드/PGP/서명/인용 제거
    t = RE_CODE.sub(" ", t)
    t = RE_PGP.sub(" ", t)
    t = RE_SIG.sub(" ", t)
    t = RE_QUOTES.sub("", t)
    # 헤더 줄 제거
    lines = []
    for ln in t.splitlines():
        ls = ln.strip()
        if ls.lower().startswith(HEADER_PREFIXES):
            continue
        lines.append(ln)
    t = "\n".join(lines)

    # URL/이메일/경로 치환 → 공백
    t = RE_URL.sub(" ", t)
    t = RE_EMAIL.sub(" ", t)
    t = RE_PATH.sub(" ", t)

    # 구두점 제거 및 다중 공백 정리
    t = RE_PUNCT.sub(" ", t)
    t = re.sub(r"\s+", " ", t).strip().lower()
    return t

# ======================
# spaCy (선택) 로더
# ======================
def _maybe_load_spacy(use_spacy: bool):
    if not use_spacy:
        return None
    try:
        import spacy
        nlp = spacy.load("en_core_web_sm", disable=["parser"])
        nlp.enable_pipe("lemmatizer")
        return nlp
    except Exception:
        # spaCy 미설치/모델 미다운로드 시 자동으로 비활성
        return None

# ======================
# 전처리(단일 문서)
# ======================
def clean_text(
    text: str,
    stopwords: Set[str],
    token_min_len: int = 3,
    use_spacy: bool = False,
    keep_pos: Set[str] = frozenset({"NOUN","ADJ","PROPN"}),
    remove_person: bool = True
) -> str:
    """
    - 먼저 20NG 노이즈 제거(strip_20ng_noise)
    - (선택) spaCy로 lemmatize + POS/NER 필터
    - 아니면 regex 토크나이즈 + stopword/길이 필터
    """
    if not isinstance(text, str):
        return ""
    t = strip_20ng_noise(text)

    nlp = _maybe_load_spacy(use_spacy)
    if nlp is not None:
        doc = nlp(t)
        kept = []
        for tok in doc:
            if tok.is_space or tok.is_punct or tok.like_num:
                continue
            lemma = tok.lemma_.lower()
            if len(lemma) < token_min_len:
                continue
            if remove_person and tok.ent_type_ == "PERSON":
                continue
            if tok.pos_ not in keep_pos:
                continue
            if lemma in stopwords:
                continue
            kept.append(lemma)
        return " ".join(kept)

    # fallback: regex 토큰화
    toks = []
    for w in RE_TOKEN.findall(t):
        w = w.lower()
        if len(w) < token_min_len:
            continue
        if w in stopwords:
            continue
        toks.append(w)
    return " ".join(toks)

# ======================
# 배치 처리
# ======================
def clean_corpus(
    texts: List[str],
    header_texts: Optional[Iterable[str]] = None,
    extra_stopwords: Optional[Iterable[str]] = None,
    token_min_len: int = 3,
    use_spacy: bool = False,
    keep_pos: Set[str] = frozenset({"NOUN","ADJ","PROPN"}),
    remove_person: bool = True
) -> List[str]:
    """
    texts: 원문 리스트
    header_texts: 헤더 전용 stopword 채굴용(대개 texts 그대로 줘도 OK)
    """
    stopwords = build_stopwords(header_texts or texts, extra_stopwords=extra_stopwords)
    return [
        clean_text(t, stopwords=stopwords, token_min_len=token_min_len,
                   use_spacy=use_spacy, keep_pos=keep_pos, remove_person=remove_person)
        for t in texts
    ]


In [11]:
from tqdm.auto import tqdm
tqdm.pandas()  # 한 번만 등록

def clean_one(doc: str, use_spacy: bool = True) -> str:
    if not isinstance(doc, str) or not doc.strip():
        return ""
    return clean_corpus([doc], use_spacy=use_spacy)[0]

USE_SPACY = True
df_20ng["cleaned_text"] = df_20ng["text"].progress_apply(
    lambda s: clean_one(s, use_spacy=USE_SPACY)
)

  0%|          | 0/18742 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [64]:
df_20ng['text'][3]

'in article wayne smith writes think it s the scsi card doing the dma transfers not the disks the scsi card can do dma transfers containing data from any of the scsi devices it is attached when it wants to an important feature of scsi is the ability to detach a device this frees the scsi bus for other devices this is typically used in a multi tasking os to start transfers on several devices while each device is seeking the data the bus is free for other commands and data transfers when the devices are ready to transfer the data they can aquire the bus and send the data on an ide bus when you start a transfer the bus is busy until the disk has seeked the data and transfered it this is typically a 10 20ms second lock out for other processes wanting the bus irrespective of transfer time'

In [12]:
df_20ng['cleaned_text'][0]

KeyError: 'cleaned_text'

### 해야할 전처리</br>
먼저 16000?개짜리 preprocessing된거 알지? 그거 열어보고 거기에 인덱스 있으면 거기에 맞춰주기
1. null 문장 삭제</br>
2. 너무 짧은 문장 필터링 및 삭제</br>
3. 직접 한번 봐보고 삭제</br>

In [66]:
df_20ng = df_20ng.dropna()

In [67]:
def drop_and_report_short(df, col="cleaned_text", min_len=20):
    """
    cleaned_text 길이가 min_len 미만인 행을 보여주고, 
    그 행들을 제거한 DataFrame을 반환합니다.
    """
    mask = df[col].fillna("").str.len() < min_len
    short_df = df[mask]

    print(f"👉 삭제 대상 행 개수: {short_df.shape[0]}")
    if not short_df.empty:
        print("=== 삭제될 행 샘플 ===")
        print(short_df[[col]].head(10))  # cleaned_text 컬럼만 샘플 출력

    cleaned_df = df[~mask]
    print(f"✅ 삭제 후 데이터프레임 크기: {cleaned_df.shape}")

    return cleaned_df, short_df

In [68]:
df_20ng, short_df = drop_and_report_short(df_20ng, col="cleaned_text", min_len=20)

👉 삭제 대상 행 개수: 507
=== 삭제될 행 샘플 ===
            cleaned_text
185                     
200    commandment peace
206                     
230      cat squid squid
268                trash
538                     
542                steve
653                     
710       guy game luigi
725  description product
✅ 삭제 후 데이터프레임 크기: (18235, 4)


In [16]:
from octis.dataset.dataset import Dataset

# 20 Newsgroups 데이터셋 다운로드 및 로드
dataset = Dataset()
dataset.fetch_dataset("20NewsGroup")

# 데이터 확인
texts = dataset.get_corpus()   # 문서 리스트
print(texts[0])                # 0번째 문서 출력


['fax', 'modem', 'card', 'sell', 'mail']


In [69]:
df_20ng.to_csv('/home/ys0660/2507Sub/textclustering/0915/stage1/data/20ng.csv')

임베딩까지 같이 뽑자

In [70]:
from __future__ import annotations

# ===== Standard Library =====
import os
import re
import math
import argparse
from typing import List, Tuple, Dict, Iterable
from collections import Counter, defaultdict
from itertools import combinations

# ===== Third-Party =====
import numpy as np
import pandas as pd
import matplotlib.pyplot as pltsS
from tqdm import tqdm
from scipy.optimize import linear_sum_assignment

# ===== scikit-learn =====
# Data
from sklearn.datasets import fetch_20newsgroups

# Text / Features
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

# Dimensionality Reduction
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.manifold import TSNE

# Clustering
from sklearn.cluster import KMeans

# Metrics
from sklearn.metrics import (
    f1_score,
    adjusted_rand_score,
    homogeneity_score,
    confusion_matrix,
    accuracy_score,
    recall_score,
    classification_report,
)

# Preprocessing
from sklearn.preprocessing import normalize

from dotenv import load_dotenv
load_dotenv()

True

In [71]:
df = df_20ng

In [72]:
cleaned_text = df['cleaned_text'].tolist()
y_true = df['label_id'].tolist()

In [73]:
from openai import OpenAI
import os

In [74]:
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [75]:
all_vecs = []
batch_size = 128  # 배치 크기 조정 가능
model_name = "text-embedding-3-small"

In [76]:
import tiktoken

enc = tiktoken.encoding_for_model("text-embedding-3-small")

MAX_TOKENS = 8192
def truncate_text(text, max_tokens=MAX_TOKENS):
    tokens = enc.encode(text)
    if len(tokens) > max_tokens:
        tokens = tokens[:max_tokens]
    return enc.decode(tokens)

In [77]:
print("[2] Call API for Embeddings...")

def get_embed_dim(model_name: str) -> int:
    name = model_name.lower()
    if "text-embedding-3-large" in name:
        return 3072
    # small/ada-002 등 기본 1536
    return 1536

def embed_texts_and_align_labels(
    texts: List[Union[str, List[str]]],   # string 또는 tokenized list
    labels: List[int],
    model: str = "text-embedding-3-small"
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, Dict[str, List[int]]]:
    """
    반환:
      X: (N, D)  - 입력 개수 그대로 유지(빈/실패는 0-벡터)
      mask_nonzero: (N,) - 0-벡터가 아닌 행(True)
      labels_clean: (M,) - mask_nonzero 적용
      texts_clean:  (M,) - mask_nonzero 적용
      logs: dict    - {'ok': [...], 'empty': [...], 'fail': [...], 'dim': D}
    """
    assert len(texts) == len(labels), f"len(texts)={len(texts)} vs len(labels)={len(labels)}"
    N = len(texts)
    D = get_embed_dim(model)

    X = np.zeros((N, D), dtype=np.float32)
    mask_nonzero = np.zeros(N, dtype=bool)

    ok_idx: List[int] = []
    empty_idx: List[int] = []
    fail_idx: List[int] = []

    for idx, raw in enumerate(tqdm(texts, desc="Embedding", total=N)):
        # --- [변경된 부분] 리스트면 문자열로 합치기 ---
        if isinstance(raw, list):
            text = " ".join(map(str, raw))   # 리스트 → 문자열
        else:
            text = str(raw)

        text = truncate_text(text.strip())  # 토큰 제한 및 공백 제거

        if not text:
            empty_idx.append(idx)
            continue
        try:
            resp = client.embeddings.create(model=model, input=text)
            vec = np.asarray(resp.data[0].embedding, dtype=np.float32)

            # 차원 안전장치
            if vec.shape[0] != D:
                print(f"[warn] idx={idx} dim change: {vec.shape[0]} != {D}. Adjusting matrix.")
                newD = vec.shape[0]
                if newD > D:
                    pad = np.zeros((N, newD - D), dtype=np.float32)
                    X = np.hstack([X, pad])
                else:
                    X = X[:, :newD]
                D = newD

            X[idx] = vec
            mask_nonzero[idx] = True
            ok_idx.append(idx)
        except Exception as e:
            print(f"[warn] idx={idx} embedding failed: {e!r}")
            fail_idx.append(idx)

    # 요약 로그
    print("\n=== Embedding Summary ===")
    print(f"Total inputs   : {N}")
    print(f"OK             : {len(ok_idx)}")
    print(f"Empty texts    : {len(empty_idx)}")
    print(f"API failures   : {len(fail_idx)}")
    print(f"Kept (non-zero): {mask_nonzero.sum()} | Dropped: {(~mask_nonzero).sum()}")
    if empty_idx[:5]:
        print(f"First empty idx: {empty_idx[:5]}")
    if fail_idx[:5]:
        print(f"First fail idx : {fail_idx[:5]}")

    labels_arr = np.asarray(labels)
    texts_arr  = np.asarray(texts, dtype=object)
    labels_clean = labels_arr[mask_nonzero]
    texts_clean  = texts_arr[mask_nonzero]

    logs = {"ok": ok_idx, "empty": empty_idx, "fail": fail_idx, "dim": D}
    return X, mask_nonzero, labels_clean, texts_clean, logs


[2] Call API for Embeddings...


In [78]:
def save_clean_pack(save_path: str, X_clean: np.ndarray,
                    labels_clean: np.ndarray, texts_clean: np.ndarray):
    Path(save_path).parent.mkdir(parents=True, exist_ok=True)
    np.savez(save_path, X=X_clean, labels=labels_clean, texts=texts_clean)
    print(f"[saved] {save_path} | X={X_clean.shape}, labels={labels_clean.shape}, texts={texts_clean.shape}")

In [79]:
from pathlib import Path

In [80]:
X, mask_nonzero, labels_clean, texts_clean, logs = embed_texts_and_align_labels(
    cleaned_text, y_true, model="text-embedding-3-small"
)
X_clean = X[mask_nonzero]  # 여기서만 마스크 적용

print("Before/After:", X.shape, "->", X_clean.shape)

# 저장
save_path = "/home/ys0660/2507Sub/textclustering/0915/stage1/01data/mine_gpt_embedding_v4.npz"
save_clean_pack(save_path, X_clean, labels_clean, texts_clean)

Embedding: 100%|██████████| 18235/18235 [1:44:28<00:00,  2.91it/s]  


=== Embedding Summary ===
Total inputs   : 18235
OK             : 18235
Empty texts    : 0
API failures   : 0
Kept (non-zero): 18235 | Dropped: 0
Before/After: (18235, 1536) -> (18235, 1536)
[saved] /home/ys0660/2507Sub/textclustering/0915/stage1/01data/mine_gpt_embedding_v4.npz | X=(18235, 1536), labels=(18235,), texts=(18235,)
